## **LLM Agents with Tools** 

In [1]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
import torch

d:\Projects\ai-math-assistant-with-LangChain-and-LangGraph\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
torch.__version__

'2.5.1+cu121'

In [11]:
device

device(type='cuda')

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

In [6]:
bits_and_bytes_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3"
)

In [12]:
llm_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    quantization_config = bits_and_bytes_config,
    
).to(device)

Loading weights: 100%|██████████| 291/291 [00:37<00:00,  7.72it/s]


In [17]:
pipe = pipeline(
    "text-generation",
    llm_model,
    tokenizer=tokenizer
)

In [18]:
pipeline=HuggingFacePipeline(
    pipeline=pipe
)

llm = ChatHuggingFace(llm=pipeline)

In [19]:
response = llm.invoke("Tell me about AI Agents")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [20]:
response.content

"<s>[INST] Tell me about AI Agents[/INST]AI agents, in the field of artificial intelligence (AI), are a key concept that represents an entity capable of performing tasks in a specific environment. They are designed to perceive their environment, take actions, and learn from the results of those actions to improve their performance over time.\n\nThere are four types of AI agents:\n\n1. Simple Reflex Agents: These agents choose their actions based solely on their current percept. They don't maintain any data structures representing the world.\n\n2. Model-Based Reflex Agents: These agents use a model of the world to determine actions. They maintain a data structure representing the world and use it to decide on actions.\n\n3. Goal-Based Agents: These agents maintain a goal and act to achieve that goal. They may use a model of the world to help them reach their goal.\n\n4. Utility-Based Agents: These agents have a utility function that assigns a numerical value to each possible world state

### Defining an add function

Now use this tool framework to create a custom tool that enables the LLM to perform basic addition.

In [21]:
from langchain_core.tools import tool

In [22]:
@tool
def add(a:int, b:int) -> int:
    """This tool is used add two integers and return the sum of two integers.

    Args:
        a (int): this is a integer value that represents the first parameter 
        b (int): this is a integer value that represents the second parameter

    Returns:
        int: this returns the sum of two integers
    """
    
    sum = a + b
    return sum

In [23]:
add.invoke({"a":5, "b":2})

7

In [24]:
@tool
def substract(a:int, b:int) -> int:
    """Subtract b from a.

    """
    
    return a-b


@tool
def multiply(a:int, b:int) -> int:
    """multiply a with b
    """
    
    return a * b

@tool
def divide(a:int, b:int) ->float:
    
    """divide a from b
    """
    
    return a/b

In [25]:
tools_map = {
    "add": add,
    "multiply": multiply,
    "divide": divide,
    "subs": substract
}

In [26]:
tools_map['divide'].invoke({"a":10, "b":4})

2.5

### Add Tools to LLM

In [27]:
tools = [add, substract, multiply, divide]

In [45]:
llm_with_tools = llm.bind_tools(tools)

### Interact with the Model

In [36]:
from langchain_core.messages import HumanMessage

In [52]:
query = "what is canda current population in year 2026 divide by 0.75"
chat_history = [HumanMessage(content=query)]

First,setup the question (user query). Then,initialize a `chat_history` array that will contain the entire conversation between user and LLM. In this chat history, you insert the `query` in a `HumanMessage` wrapper that tells LangChain and the model: "This message came from the user."


### Invoke with the Model

In [53]:
response_1 = llm_with_tools.invoke(chat_history)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [54]:
response_1

AIMessage(content="<s>[INST] what is canda current population in year 2026 divide by 0.75[/INST]To find the current population of Canada in the year 2026 and then divide it by 0.75, we first need to estimate the population for 2026. As of my last update, the population of Canada was approximately 38.06 million (July 2021 estimate). To find the population in 2026, we can use a growth rate. I'll use a conservative growth rate of 0.8% per year.\n\nPopulation in 2026 = Population in 2021 * (1 + Growth rate)^(Number of years)\nPopulation in 2026 = 38.06 million * (1 + 0.008)^5\n\nNow, let's calculate the population in 2026:\nPopulation in 2026 ≈ 41.39 million\n\nNow, we divide this number by 0.75:\nPopulation in 2026 / 0.75 ≈ 55.26 million\n\nThis result is an estimate of the population of Canada in 202", additional_kwargs={}, response_metadata={}, id='run--019f40e4-07d4-7690-b935-2eae02c3e3d3-0')

In [55]:
chat_history.append(response_1)

In [56]:
response_1.tool_calls

[]

In [57]:
print(tokenizer.chat_template)

{%- if messages[0]["role"] == "system" %}
    {%- set system_message = messages[0]["content"] %}
    {%- set loop_messages = messages[1:] %}
{%- else %}
    {%- set loop_messages = messages %}
{%- endif %}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}
{%- set user_messages = loop_messages | selectattr("role", "equalto", "user") | list %}

{#- This block checks for alternating user/assistant messages, skipping tool calling messages #}
{%- set ns = namespace() %}
{%- set ns.index = 0 %}
{%- for message in loop_messages %}
    {%- if not (message.role == "tool" or message.role == "tool_results" or (message.tool_calls is defined and message.tool_calls is not none)) %}
        {%- if (message["role"] == "user") != (ns.index % 2 == 0) %}
            {{- raise_exception("After the optional system message, conversation roles must alternate user/assistant/user/assistant/...") }}
        {%- endif %}
        {%- set ns.index = ns.index + 1 %}
    {%- endif %}
{%- endfor